In [ ]:
pip install pandas scipy statsmodels

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# BLIP-OLIVE HUMAN-BASED EVALUATION
# Descriptive Statistics + Friedman Test + Kendall's W
# + Wilcoxon Post-hoc Tests + Holm Correction
# ============================================================

import pandas as pd
import numpy as np

from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. LOAD DATA
# ============================================================

file_path = "/content/drive/MyDrive/T-Shirt Sign Up2.csv"

df = pd.read_csv(file_path)

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Dataset shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")


# ============================================================
# 2. SELECT THE 8 SECTION-1 EVALUATION CRITERIA
# ============================================================
#
# In your CSV:
# column 0 = timestamp / metadata
# columns 1 to 8 = the eight BLIP-OLIVE evaluation criteria
#
# Using iloc avoids KeyError caused by the long Google Forms
# column names.
# ============================================================

data = df.iloc[:, 1:9].copy()


# Give the columns short, clean names
data.columns = [
    "Clarity",
    "Consistency",
    "Treatment",
    "DecisionSupport",
    "Completeness",
    "Coherence",
    "Trustworthiness",
    "Relevance"
]


print("\n" + "=" * 70)
print("SELECTED CRITERIA")
print("=" * 70)

print(data.columns.tolist())


# ============================================================
# 3. CONVERT LIKERT RESPONSES TO NUMERIC VALUES
# ============================================================
#
# Google Forms responses may look like:
#
# 5 (Very High)
# 4 (High)
# 3 (Moderate)
# 2 (Low)
# 1 (Very Low)
#
# We extract the first digit and convert it to numeric.
# ============================================================

for col in data.columns:

    data[col] = (
        data[col]
        .astype("string")
        .str.extract(r"^\s*([1-5])", expand=False)
    )

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )


print("\n" + "=" * 70)
print("NUMERIC DATA BEFORE REMOVING MISSING VALUES")
print("=" * 70)

print(data)


# ============================================================
# 4. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

print(data.isna().sum())


# ============================================================
# 5. KEEP COMPLETE PARTICIPANTS
# ============================================================
#
# Friedman requires paired/repeated observations.
# Therefore, a participant must have a valid response for
# all 8 criteria included in the test.
# ============================================================

data_complete = data.dropna().copy()


print("\n" + "=" * 70)
print("COMPLETE CASES")
print("=" * 70)

print(f"Original number of records : {len(df)}")
print(f"Complete participants      : {len(data_complete)}")
print(f"Excluded records           : {len(df) - len(data_complete)}")


print("\nData used for Friedman:")
print(data_complete)


# ============================================================
# 6. DESCRIPTIVE STATISTICS
# ============================================================

results = []

for column in data_complete.columns:

    x = data_complete[column]

    n_valid = x.count()

    mean = x.mean()

    sd = x.std(ddof=1)

    median = x.median()

    q1 = x.quantile(0.25)

    q3 = x.quantile(0.75)

    iqr = q3 - q1

    # Favorable evaluation = rating 4 or 5
    positive_n = (x >= 4).sum()

    positive_percentage = (
        positive_n / n_valid
    ) * 100


    results.append({

        "Criterion": column,

        "N": n_valid,

        "Mean": mean,

        "SD": sd,

        "Median": median,

        "Q1": q1,

        "Q3": q3,

        "IQR": iqr,

        "Positive_N": positive_n,

        "Positive_4_5_%": positive_percentage
    })


descriptive = pd.DataFrame(results)


print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)


print(
    descriptive.round({
        "Mean": 2,
        "SD": 2,
        "Median": 2,
        "Q1": 2,
        "Q3": 2,
        "IQR": 2,
        "Positive_4_5_%": 1
    }).to_string(index=False)
)


# ============================================================
# 7. RESPONSE FREQUENCIES
# ============================================================
#
# Useful for reporting the distribution of 1–5 responses.
# ============================================================

frequency_results = []

for column in data_complete.columns:

    counts = (
        data_complete[column]
        .value_counts()
        .reindex([1, 2, 3, 4, 5], fill_value=0)
    )

    n = len(data_complete[column])


    frequency_results.append({

        "Criterion": column,

        "Rating_1_N": counts[1],
        "Rating_2_N": counts[2],
        "Rating_3_N": counts[3],
        "Rating_4_N": counts[4],
        "Rating_5_N": counts[5],

        "Rating_1_%": counts[1] / n * 100,
        "Rating_2_%": counts[2] / n * 100,
        "Rating_3_%": counts[3] / n * 100,
        "Rating_4_%": counts[4] / n * 100,
        "Rating_5_%": counts[5] / n * 100
    })


frequencies = pd.DataFrame(frequency_results)


print("\n" + "=" * 70)
print("LIKERT RESPONSE FREQUENCIES")
print("=" * 70)

print(
    frequencies.round(1).to_string(index=False)
)


# ============================================================
# 8. FRIEDMAN TEST
# ============================================================
#
# H0:
# The distributions/ranks of the eight evaluation criteria
# are equivalent.
#
# H1:
# At least one criterion differs.
# ============================================================

friedman_stat, friedman_p = friedmanchisquare(
    *[
        data_complete[col].to_numpy()
        for col in data_complete.columns
    ]
)


df_friedman = len(data_complete.columns) - 1


print("\n" + "=" * 70)
print("FRIEDMAN TEST")
print("=" * 70)

print(f"Chi-square statistic = {friedman_stat:.4f}")
print(f"Degrees of freedom   = {df_friedman}")
print(f"p-value              = {friedman_p:.8f}")


if friedman_p < 0.001:

    print("Result: statistically significant (p < 0.001)")

elif friedman_p < 0.05:

    print("Result: statistically significant (p < 0.05)")

else:

    print("Result: not statistically significant (p >= 0.05)")


# ============================================================
# 9. KENDALL'S W EFFECT SIZE
# ============================================================
#
# W = chi-square / [N × (k - 1)]
#
# N = number of participants
# k = number of criteria
#
# W ranges from 0 to 1.
# ============================================================

n = len(data_complete)

k = len(data_complete.columns)

kendall_w = friedman_stat / (
    n * (k - 1)
)


print("\n" + "=" * 70)
print("KENDALL'S W EFFECT SIZE")
print("=" * 70)

print(f"N                  = {n}")
print(f"Number of criteria = {k}")
print(f"Kendall's W        = {kendall_w:.4f}")


# Descriptive interpretation only
if kendall_w < 0.10:

    effect_interpretation = "very small"

elif kendall_w < 0.30:

    effect_interpretation = "small"

elif kendall_w < 0.50:

    effect_interpretation = "moderate"

else:

    effect_interpretation = "large"


print(
    f"Approximate effect-size interpretation: "
    f"{effect_interpretation}"
)


# ============================================================
# 10. POST-HOC WILCOXON SIGNED-RANK TESTS
# ============================================================
#
# Friedman only tells us that an overall difference exists.
#
# Pairwise Wilcoxon signed-rank tests investigate which
# criteria differ.
# ============================================================

pairwise_results = []

columns = data_complete.columns.tolist()


for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        criterion1 = columns[i]

        criterion2 = columns[j]


        x = data_complete[criterion1].to_numpy()

        y = data_complete[criterion2].to_numpy()


        # Paired differences
        differences = x - y

        nonzero_differences = differences[
            differences != 0
        ]


        # If all paired values are identical,
        # Wilcoxon cannot be calculated normally.
        if len(nonzero_differences) == 0:

            statistic = 0.0

            p_value = 1.0

        else:

            statistic, p_value = wilcoxon(
                x,
                y,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )


        pairwise_results.append({

            "Criterion_1": criterion1,

            "Criterion_2": criterion2,

            "Wilcoxon_W": statistic,

            "Raw_p": p_value

        })


posthoc = pd.DataFrame(pairwise_results)


# ============================================================
# 11. HOLM CORRECTION
# ============================================================
#
# There are:
#
# C(8,2) = 28 pairwise comparisons.
#
# Therefore, multiple-comparison correction is required.
#
# Holm correction controls the family-wise error rate.
# ============================================================

reject, corrected_p, _, _ = multipletests(

    posthoc["Raw_p"],

    alpha=0.05,

    method="holm"
)


posthoc["Holm_p"] = corrected_p

posthoc["Significant"] = reject


print("\n" + "=" * 70)
print("POST-HOC WILCOXON TESTS + HOLM CORRECTION")
print("=" * 70)


posthoc_display = posthoc.copy()


posthoc_display["Raw_p"] = (
    posthoc_display["Raw_p"].round(5)
)

posthoc_display["Holm_p"] = (
    posthoc_display["Holm_p"].round(5)
)


print(
    posthoc_display.to_string(index=False)
)


# ============================================================
# 12. SHOW ONLY SIGNIFICANT POST-HOC COMPARISONS
# ============================================================

significant = posthoc[
    posthoc["Significant"]
].copy()


print("\n" + "=" * 70)
print("SIGNIFICANT PAIRWISE DIFFERENCES")
print("=" * 70)


if significant.empty:

    print(
        "No pairwise comparison remained statistically "
        "significant after Holm correction."
    )

else:

    significant["Raw_p"] = (
        significant["Raw_p"].round(5)
    )

    significant["Holm_p"] = (
        significant["Holm_p"].round(5)
    )

    print(
        significant.to_string(index=False)
    )


# ============================================================
# 13. MEAN RANKS
# ============================================================
#
# Friedman works with ranks.
# Mean ranks can help describe which criteria tended to
# receive higher/lower ratings.
# ============================================================

ranked_data = data_complete.rank(
    axis=1,
    method="average"
)


mean_ranks = (
    ranked_data
    .mean(axis=0)
    .sort_values(ascending=False)
    .reset_index()
)


mean_ranks.columns = [
    "Criterion",
    "Mean_Rank"
]


print("\n" + "=" * 70)
print("FRIEDMAN MEAN RANKS")
print("=" * 70)

print(
    mean_ranks.round(3).to_string(index=False)
)


# ============================================================
# 14. CREATE SUMMARY OF FRIEDMAN TEST
# ============================================================

friedman_summary = pd.DataFrame({

    "N": [n],

    "Number_of_Criteria": [k],

    "Chi_Square": [friedman_stat],

    "df": [df_friedman],

    "p_value": [friedman_p],

    "Kendalls_W": [kendall_w]

})


# ============================================================
# 15. SAVE ALL RESULTS
# ============================================================

descriptive.to_csv(
    "/content/descriptive_statistics.csv",
    index=False
)


frequencies.to_csv(
    "/content/likert_frequencies.csv",
    index=False
)


friedman_summary.to_csv(
    "/content/friedman_test_results.csv",
    index=False
)


mean_ranks.to_csv(
    "/content/friedman_mean_ranks.csv",
    index=False
)


posthoc.to_csv(
    "/content/wilcoxon_holm_results.csv",
    index=False
)


significant.to_csv(
    "/content/significant_pairwise_results.csv",
    index=False
)


print("\n" + "=" * 70)
print("RESULTS SAVED SUCCESSFULLY")
print("=" * 70)

print(
    """
Files created:

1. descriptive_statistics.csv
2. likert_frequencies.csv
3. friedman_test_results.csv
4. friedman_mean_ranks.csv
5. wilcoxon_holm_results.csv
6. significant_pairwise_results.csv
"""
)

DATASET INFORMATION
Dataset shape: (21, 17)

Columns:
0: Timestamp
1: Criterion 1: Clarity and Readability
Associated Question:  How clear and easy to understand is the explanation provided by the BLIP-OLIVE? 
2: Criterion 2: Symptom–disease consistency
Associated Question:   How consistent is the predicted disease with the symptoms identified by the BLIP-OLIVE? 
3: Criterion 3:  Usefulness of treatment recommendations 
Associated Question:  How useful and relevant are the treatment recommendations provided by the BLIP-OLIVE? 
4: Criterion 4: Diagnosis and decision-making support  
Associated Question:  How useful is the BLIP-OLIVE overall in supporting disease diagnosis and decision-making? 
5: Criterion 5: Completeness of the Explanation
Associated Question:  How complete is the explanation provided by the BLIP-OLIVE in describing the identified symptoms and predicted disease?
6: Criterion 6: Coherence of the Generated Output
Associated Question:  How coherent and logically structure

In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. LOAD DATA
# ============================================================

file_path = "/content/drive/MyDrive/T-Shirt Sign Up2.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns):
    print(i, ":", col)


# ============================================================
# 2. AUTOMATICALLY FIND SECTION 2 COLUMNS
# ============================================================

def find_column(keyword):
    matches = [
        col for col in df.columns
        if keyword.lower() in col.lower()
    ]

    if len(matches) == 0:
        raise ValueError(
            f"No column found containing: '{keyword}'"
        )

    if len(matches) > 1:
        print(f"\nWarning: multiple columns found for '{keyword}':")
        for m in matches:
            print("  ", m)

    return matches[0]


criteria = {
    "Overall Added Value":
        find_column("overall added value"),

    "Practical Decision Support":
        find_column("real diagnostic decisions"),

    "Intention to Use":
        find_column("interested would you be in using"),

    "Recommendation Intention":
        find_column("recommend BLIP-OLIVE")
}


print("\n========================================")
print("SECTION 2 COLUMNS DETECTED")
print("========================================")

for name, col in criteria.items():
    print(f"\n{name}:")
    print(col)


# ============================================================
# 3. CONVERT RESPONSES TO NUMERIC VALUES
# ============================================================
#
# This works for:
# 5
# 5 (Much better)
# 4 (Somewhat better)
# 5 (Extremely)
# etc.
# ============================================================

data = pd.DataFrame()

for short_name, column_name in criteria.items():

    data[short_name] = pd.to_numeric(
        df[column_name]
        .astype(str)
        .str.extract(r'^\s*([1-5])')[0],
        errors="coerce"
    )


# Keep only participants who answered ALL four questions
data_complete = data.dropna()

print("\n========================================")
print("COMPLETE PARTICIPANTS")
print("========================================")

print("N =", len(data_complete))

print("\nData used:")
print(data_complete)


# ============================================================
# 4. DESCRIPTIVE STATISTICS
# ============================================================

results = []

for column in data_complete.columns:

    x = data_complete[column]

    n = len(x)
    mean = x.mean()
    sd = x.std(ddof=1)
    median = x.median()

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1

    positive_n = (x >= 4).sum()
    positive_pct = positive_n / n * 100

    results.append({
        "Criterion": column,
        "N": n,
        "Mean": mean,
        "SD": sd,
        "Median": median,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Positive_N": positive_n,
        "Positive_4_5_%": positive_pct
    })


descriptive = pd.DataFrame(results)

print("\n========================================")
print("DESCRIPTIVE STATISTICS - SECTION 2")
print("========================================")

print(
    descriptive.round({
        "Mean": 2,
        "SD": 2,
        "Median": 2,
        "Q1": 2,
        "Q3": 2,
        "IQR": 2,
        "Positive_4_5_%": 1
    }).to_string(index=False)
)


# ============================================================
# 5. RESPONSE DISTRIBUTIONS
# ============================================================

print("\n========================================")
print("RESPONSE DISTRIBUTIONS")
print("========================================")

for column in data_complete.columns:

    print(f"\n{column}")

    counts = (
        data_complete[column]
        .value_counts()
        .sort_index()
    )

    for score in range(1, 6):

        count = counts.get(score, 0)
        percentage = count / len(data_complete) * 100

        print(
            f"Score {score}: "
            f"{count} participants "
            f"({percentage:.1f}%)"
        )


# ============================================================
# 6. FRIEDMAN TEST
# ============================================================

friedman_stat, friedman_p = friedmanchisquare(
    *[
        data_complete[col].values
        for col in data_complete.columns
    ]
)

print("\n========================================")
print("FRIEDMAN TEST - SECTION 2")
print("========================================")

print(f"Chi-square = {friedman_stat:.4f}")
print(f"df         = {len(data_complete.columns) - 1}")
print(f"p-value    = {friedman_p:.8f}")


# ============================================================
# 7. KENDALL'S W
# ============================================================

n = len(data_complete)
k = len(data_complete.columns)

kendall_w = friedman_stat / (n * (k - 1))

print("\n========================================")
print("EFFECT SIZE")
print("========================================")

print(f"N = {n}")
print(f"Number of criteria = {k}")
print(f"Kendall's W = {kendall_w:.4f}")


# ============================================================
# 8. FRIEDMAN MEAN RANKS
# ============================================================

# Rank the 4 scores WITHIN each participant.
# Higher score -> higher rank.

ranks = data_complete.rank(
    axis=1,
    method="average",
    ascending=True
)

mean_ranks = ranks.mean(axis=0)

mean_rank_table = pd.DataFrame({
    "Criterion": mean_ranks.index,
    "Mean_Rank": mean_ranks.values
})

mean_rank_table = mean_rank_table.sort_values(
    "Mean_Rank",
    ascending=False
)

print("\n========================================")
print("FRIEDMAN MEAN RANKS")
print("========================================")

print(
    mean_rank_table
    .round({"Mean_Rank": 3})
    .to_string(index=False)
)


# ============================================================
# 9. POST-HOC WILCOXON TESTS
# ============================================================
#
# Only interpret post-hoc comparisons if the Friedman
# omnibus test is statistically significant.
# ============================================================

columns = data_complete.columns.tolist()

pairwise_results = []

for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        c1 = columns[i]
        c2 = columns[j]

        x = data_complete[c1]
        y = data_complete[c2]

        differences = x - y

        # All differences equal zero
        if np.all(differences == 0):

            statistic = 0.0
            p = 1.0

        else:

            statistic, p = wilcoxon(
                x,
                y,
                alternative="two-sided",
                zero_method="wilcox"
            )

        pairwise_results.append({
            "Criterion_1": c1,
            "Criterion_2": c2,
            "Wilcoxon_W": statistic,
            "Raw_p": p
        })


posthoc = pd.DataFrame(pairwise_results)


# ============================================================
# 10. HOLM CORRECTION
# ============================================================

reject, corrected_p, _, _ = multipletests(
    posthoc["Raw_p"],
    alpha=0.05,
    method="holm"
)

posthoc["Holm_p"] = corrected_p
posthoc["Significant"] = reject


print("\n========================================")
print("POST-HOC WILCOXON + HOLM")
print("========================================")

print(
    posthoc.round({
        "Wilcoxon_W": 3,
        "Raw_p": 5,
        "Holm_p": 5
    }).to_string(index=False)
)


# ============================================================
# 11. SIGNIFICANT PAIRWISE COMPARISONS
# ============================================================

significant = posthoc[
    posthoc["Significant"] == True
]

print("\n========================================")
print("SIGNIFICANT PAIRWISE DIFFERENCES")
print("========================================")

if len(significant) == 0:

    print(
        "No pairwise comparison remained "
        "significant after Holm correction."
    )

else:

    print(
        significant.round({
            "Wilcoxon_W": 3,
            "Raw_p": 5,
            "Holm_p": 5
        }).to_string(index=False)
    )


# ============================================================
# 12. SAVE RESULTS
# ============================================================

descriptive.to_csv(
    "section2_descriptive_statistics.csv",
    index=False
)

mean_rank_table.to_csv(
    "section2_friedman_mean_ranks.csv",
    index=False
)

posthoc.to_csv(
    "section2_wilcoxon_holm_results.csv",
    index=False
)

print("\n========================================")
print("RESULTS SAVED SUCCESSFULLY")
print("========================================")

Dataset shape: (21, 17)

Columns:
0 : Timestamp
1 : Criterion 1: Clarity and Readability
Associated Question:  How clear and easy to understand is the explanation provided by the BLIP-OLIVE? 
2 : Criterion 2: Symptom–disease consistency
Associated Question:   How consistent is the predicted disease with the symptoms identified by the BLIP-OLIVE? 
3 : Criterion 3:  Usefulness of treatment recommendations 
Associated Question:  How useful and relevant are the treatment recommendations provided by the BLIP-OLIVE? 
4 : Criterion 4: Diagnosis and decision-making support  
Associated Question:  How useful is the BLIP-OLIVE overall in supporting disease diagnosis and decision-making? 
5 : Criterion 5: Completeness of the Explanation
Associated Question:  How complete is the explanation provided by the BLIP-OLIVE in describing the identified symptoms and predicted disease?
6 : Criterion 6: Coherence of the Generated Output
Associated Question:  How coherent and logically structured is the info

In [ ]:
import pandas as pd
import re
from collections import Counter

# ============================================================
# 1. LOAD DATA
# ============================================================

file_path = "/content/drive/MyDrive/T-Shirt Sign Up2.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

# Section 3 = columns 13–16
section3 = df.iloc[:, 13:17].copy()

section3.columns = [
    "Human-like Assistance",
    "Clarity and Technical Complexity",
    "Impact on Current Practice",
    "Generalization to Other Crops"
]

# Remove completely empty Section 3 rows
section3 = section3.dropna(how="all")

print("\nNumber of respondents available for Section 3:")
print(len(section3))


# ============================================================
# 2. CLEAN TEXT
# ============================================================

def clean_text(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)

    if x == "":
        return None

    return x

for col in section3.columns:
    section3[col] = section3[col].apply(clean_text)


# ============================================================
# 3. NUMBER OF VALID RESPONSES PER QUESTION
# ============================================================

print("\n" + "=" * 60)
print("VALID RESPONSES PER QUESTION")
print("=" * 60)

for col in section3.columns:
    n = section3[col].notna().sum()
    print(f"{col}: {n}")


# ============================================================
# 4. PRINT ORIGINAL RESPONSES
# ============================================================

print("\n" + "=" * 60)
print("ORIGINAL QUALITATIVE RESPONSES")
print("=" * 60)

for col in section3.columns:

    print("\n" + "-" * 60)
    print(col.upper())
    print("-" * 60)

    responses = section3[col].dropna()

    for i, response in enumerate(responses, start=1):
        print(f"{i}. {response}")


# ============================================================
# 5. SIMPLE RESPONSE-ORIENTATION CODING
# ============================================================
# This is only a first-pass coding aid.
# The original responses must still be inspected manually.

def classify_orientation(text):

    if text is None:
        return "Missing"

    t = text.lower().strip()

    # Negative expressions checked first
    negative_patterns = [
        r"^no\b",
        r"^not\b",
        r"don't\b",
        r"do not\b",
        r"didn't\b",
        r"did not\b",
        r"not really",
        r"nope"
    ]

    positive_patterns = [
        r"^yes\b",
        r"^yeah\b",
        r"^sure\b",
        r"definitely",
        r"absolutely"
    ]

    uncertain_patterns = [
        r"maybe",
        r"perhaps",
        r"possibly",
        r"depends",
        r"not sure",
        r"somewhat"
    ]

    for pattern in uncertain_patterns:
        if re.search(pattern, t):
            return "Conditional/Mixed"

    for pattern in negative_patterns:
        if re.search(pattern, t):
            return "Negative/No"

    for pattern in positive_patterns:
        if re.search(pattern, t):
            return "Positive/Yes"

    return "Other/Needs manual coding"


for col in section3.columns:

    coded_col = col + " - Orientation"
    section3[coded_col] = section3[col].apply(classify_orientation)

    print("\n" + "=" * 60)
    print(col.upper())
    print("=" * 60)

    counts = section3[coded_col].value_counts()

    valid_n = section3[col].notna().sum()

    for category, count in counts.items():

        if category == "Missing":
            continue

        pct = (count / valid_n) * 100

        print(
            f"{category}: "
            f"{count}/{valid_n} ({pct:.1f}%)"
        )


# ============================================================
# 6. WORD FREQUENCIES
# ============================================================
# Exploratory only — useful for identifying possible themes.

stop_words = {
    "the", "a", "an", "and", "or", "to", "of", "in", "is",
    "it", "that", "this", "was", "were", "be", "for", "with",
    "on", "as", "i", "you", "we", "they", "my", "our", "your",
    "yes", "no", "not", "but", "so", "very", "would", "could",
    "can", "do", "did", "have", "has", "had"
}

print("\n" + "=" * 60)
print("MOST FREQUENT WORDS")
print("=" * 60)

for col in section3.columns[:4]:

    words = []

    for response in section3[col].dropna():

        tokens = re.findall(r"\b[a-zA-Z]+\b", response.lower())

        tokens = [
            word for word in tokens
            if word not in stop_words and len(word) > 2
        ]

        words.extend(tokens)

    frequencies = Counter(words).most_common(15)

    print(f"\n{col}")

    for word, count in frequencies:
        print(f"{word}: {count}")


# ============================================================
# 7. SAVE CLEANED/CODED DATA
# ============================================================

section3.to_csv(
    "/content/section3_qualitative_analysis.csv",
    index=False
)

print("\n" + "=" * 60)
print("ANALYSIS SAVED")
print("=" * 60)

print(
    "/content/section3_qualitative_analysis.csv"
)

Dataset shape: (21, 17)

Number of respondents available for Section 3:
20

VALID RESPONSES PER QUESTION
Human-like Assistance: 19
Clarity and Technical Complexity: 19
Impact on Current Practice: 18
Generalization to Other Crops: 20

ORIGINAL QUALITATIVE RESPONSES

------------------------------------------------------------
HUMAN-LIKE ASSISTANCE
------------------------------------------------------------
1. Yes, because the explanations were clear, natural, and easy to follow
2. Yes, the explanations were clear and easy to understand, without being too technical
3. Yes, the explanations were straightforward, easy to follow, and not overly technical
4. no, i felt it is a black box
5. yes, explanations are useful
6. Yes. the explanation gives more details about causes, treatement, etc.
7. Yes
8. yes, it ensures human assistant
9. Yes
10. Yes
11. Yes
12. Yes, largely. The reasoning was visible, so I could question it instead of just accepting output.
13. Yes
14. Yes
15. Yes
16. yes, it 